In [5]:
import datetime

# 天干和地支列表
TIAN_GAN = ['甲', '乙', '丙', '丁', '戊', '己', '庚', '辛', '壬', '癸']
DI_ZHI = ['子', '丑', '寅', '卯', '辰', '巳', '午', '未', '申', '酉', '戌', '亥']

# 月柱地支与节气对应（近似公历日期，调整以确保 1 月 1 日为子月）
MONTH_DI_ZHI = {
    (12, 7): '子',  # 大雪
    (1, 6): '丑',   # 小寒
    (2, 4): '寅',   # 立春
    (3, 6): '卯',   # 惊蛰
    (4, 5): '辰',   # 清明
    (5, 6): '巳',   # 立夏
    (6, 6): '午',   # 芒种
    (7, 7): '未',   # 小暑
    (8, 8): '申',   # 立秋
    (9, 8): '酉',   # 白露
    (10, 8): '戌',  # 寒露
    (11, 8): '亥'   # 立冬
}

# 月柱天干表，根据年份天干
MONTH_TIAN_GAN = {
    ('甲', '己'): ['丙', '丁', '戊', '庚', '壬', '乙', '戊', '辛', '戊', '壬', '癸', '己'],
    ('乙', '庚'): ['戊', '己', '庚', '壬', '癸', '丁', '庚', '癸', '庚', '甲', '乙', '辛'],
    ('丙', '辛'): ['庚', '辛', '壬', '甲', '丙', '己', '壬', '乙', '壬', '戊', '己', '癸'],
    ('丁', '壬'): ['壬', '癸', '甲', '丙', '戊', '辛', '甲', '丁', '甲', '庚', '辛', '乙'],
    ('戊', '癸'): ['甲', '乙', '丙', '戊', '庚', '癸', '丙', '己', '丙', '壬', '癸', '丁']
}

# 时柱地支与时间段对应
HOUR_DI_ZHI = {
    (23, 1): '子', (1, 3): '丑', (3, 5): '寅', (5, 7): '卯', (7, 9): '辰',
    (9, 11): '巳', (11, 13): '午', (13, 15): '未', (15, 17): '申', (17, 19): '酉',
    (19, 21): '戌', (21, 23): '亥'
}

# 时柱天干表，根据日干
HOUR_TIAN_GAN = {
    ('甲', '己'): ['甲', '乙', '丙', '丁', '戊', '己', '庚', '辛', '壬', '癸', '甲', '乙'],
    ('乙', '庚'): ['丙', '丁', '戊', '己', '庚', '辛', '壬', '癸', '甲', '乙', '丙', '丁'],
    ('丙', '辛'): ['戊', '己', '庚', '辛', '壬', '癸', '甲', '乙', '丙', '丁', '戊', '己'],
    ('丁', '壬'): ['庚', '辛', '壬', '癸', '甲', '乙', '丙', '丁', '戊', '己', '庚', '辛'],
    ('戊', '癸'): ['壬', '癸', '甲', '乙', '丙', '丁', '戊', '己', '庚', '辛', '壬', '癸']
}

def get_year_ganzhi(year):
    """计算年份天干地支"""
    n = (year - 3) % 60
    if n == 0:
        n = 60
    tian_gan = TIAN_GAN[(n - 1) % 10]
    di_zhi = DI_ZHI[(n - 1) % 12]
    return tian_gan, di_zhi

def get_month_ganzhi(year, month, day):
    """计算月柱天干地支（基于近似节气日期）"""
    di_zhi = '子'  # 默认子月（大雪）
    for (m, d), dz in MONTH_DI_ZHI.items():
        if (month == m and day >= d) or (month == m + 1) or (month == 1 and m == 12):
            di_zhi = dz
            break
    year_tian_gan, _ = get_year_ganzhi(year)
    for (tg1, tg2), tg_list in MONTH_TIAN_GAN.items():
        if year_tian_gan in (tg1, tg2):
            month_idx = list(MONTH_DI_ZHI.values()).index(di_zhi)
            tian_gan = tg_list[month_idx]
            break
    else:
        tian_gan = '甲'  # 默认
    return tian_gan, di_zhi

def get_day_ganzhi(year, month, day):
    """计算日柱天干地支（基于2000年1月1日为戊午，序号35）"""
    base_date = datetime.datetime(2000, 1, 1)
    target_date = datetime.datetime(year, month, day)
    days_diff = (target_date - base_date).days
    n = (35 + days_diff) % 60  # 2000年1月1日是戊午（35）
    if n == 0:
        n = 60
    tian_gan = TIAN_GAN[(n - 1) % 10]
    di_zhi = DI_ZHI[(n - 1) % 12]
    return tian_gan, di_zhi

def get_hour_ganzhi(hour, day_tian_gan):
    """计算时柱天干地支"""
    for (start, end), di_zhi in HOUR_DI_ZHI.items():
        if start <= hour < end or (start == 23 and hour >= 23):
            break
    else:
        di_zhi = '子'  # 默认子时
    for (tg1, tg2), tg_list in HOUR_TIAN_GAN.items():
        if day_tian_gan in (tg1, tg2):
            hour_idx = list(HOUR_DI_ZHI.values()).index(di_zhi)
            tian_gan = tg_list[hour_idx]
            break
    else:
        tian_gan = '甲'  # 默认
    return tian_gan, di_zhi

def calculate_bazi(year, month, day, hour):
    """主函数：计算八字"""
    year_tian_gan, year_di_zhi = get_year_ganzhi(year)
    month_tian_gan, month_di_zhi = get_month_ganzhi(year, month, day)
    day_tian_gan, day_di_zhi = get_day_ganzhi(year, month, day)
    hour_tian_gan, hour_di_zhi = get_hour_ganzhi(hour, day_tian_gan)

    return (
        f"{year_tian_gan}{year_di_zhi}",
        f"{month_tian_gan}{month_di_zhi}",
        f"{day_tian_gan}{day_di_zhi}",
        f"{hour_tian_gan}{hour_di_zhi}"
    )

# 示例使用
if __name__ == "__main__":
    # 测试 2000 年 1 月 1 日 12:00
    year, month, day, hour = 1988, 10, 12, 10
    year_pillar, month_pillar, day_pillar, hour_pillar = calculate_bazi(year, month, day, hour)
    print(f"八字: {year_pillar} {month_pillar} {day_pillar} {hour_pillar}")

八字: 戊辰 壬酉 庚辰 辛巳
